<a href="https://colab.research.google.com/github/maierav/ai_oscp_neuro/blob/main/notebooks/rf_sanity_check_three_modalities.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Receptive-field sanity check across three recording scales

A **confidence-building** notebook for the [OpenScope Community Predictive
Processing](https://allenneuraldynamics.github.io/openscope-community-predictive-processing/)
dataset. Before running any sophisticated mismatch/prediction-error analysis, we
verify that the whole pipeline — *stream NWB → align to stimulus trials → extract
neural response → build a receptive field (RF)* — produces sensible, retinotopically
localized RFs in **all three modalities**:

| Modality | DANDI | Signal | Spatial scale |
|---|---|---|---|
| Neuropixels ecephys | [001637](https://dandiarchive.org/dandiset/001637) | spike rate | area / layer |
| Mesoscope 2-photon | [001768](https://dandiarchive.org/dandiset/001768) | ΔF/F (jGCaMP8s, soma) | single cell |
| SLAP2 | [001424](https://dandiarchive.org/dandiset/001424) | ΔF/F (iGluSnFR, glutamate) | dendrite |

RF mapping is an ideal check because the expected answer is known: a real visual
neuron should respond to a **compact, contiguous patch of visual space**. If the
RFs come out as localized hotspots, the streaming, trial-alignment and
response-extraction code are all working.

**Runtime:** CPU is fine. Each session streams several GB over HTTP via `remfile`
(no full download). The whole notebook runs in ~5–8 min. Open in Colab and run top
to bottom.

In [ ]:
#@title Install dependencies
!pip -q install pynwb h5py remfile requests pandas numpy matplotlib

In [ ]:
#@title Streaming helper — resolve a DANDI asset to a signed S3 URL and open it
import h5py, remfile, requests, numpy as np, pandas as pd

def s3_url(dandiset, asset_id, version="draft"):
    """Follow the DANDI download redirect to the (signed, expiring) S3 URL."""
    r = requests.get(
        f"https://api.dandiarchive.org/api/dandisets/{dandiset}/versions/{version}/assets/{asset_id}/download/",
        allow_redirects=False, timeout=30)
    return r.headers["Location"]

def open_nwb(dandiset, asset_id):
    """Open a remote NWB for lazy, chunked HTTP reads (no full download)."""
    return h5py.File(remfile.File(s3_url(dandiset, asset_id)), "r")

def col(group, name):
    """Read an intervals/units column, decoding byte-strings to str."""
    v = group[name][:]
    return np.array([x.decode() if isinstance(x, bytes) else x for x in v])

print("helpers ready")

## 1 · Neuropixels ecephys — RFs from spikes

We use one standard-oddball session that also carries a `RF mapping` block. The RF
stimulus is a **9×9 grid of patch gratings** (azimuth/elevation ±40° in 10° steps),
each shown 0.25 s. For every visual unit we count spikes in a short post-onset
window (30–200 ms), subtract a pre-onset baseline, and average per screen position.

**Selecting examples honestly.** Mouse V1/LM receptive fields span roughly **15–30°**
— they are *not* single pixels. A naïve `peak/std` ("SNR") ranking is biased toward
one-pixel maps (a broad RF has high spatial std → low score), so we instead fit a
**2-D Gaussian** to each map and rank by fit quality (R²), which also gives the RF
width (FWHM) in degrees. Units are assigned to a CCF area/layer via the *corrected*
per-probe electrode mapping.

In [ ]:
#@title Ecephys RF — session sub-830851 (standard oddball, CCF-labelled)
ECE_ASSET = "9b9e8abe-7b43-47f1-b8e1-4114f87898a1"   # sub-830851, 2026-03-17
fh = open_nwb("001637", ECE_ASSET)

# --- RF-mapping stimulus grid ---
g = fh["intervals"]["RF mapping_presentations"]
X = col(g, "X").astype(float); Y = col(g, "Y").astype(float)
t0 = g["start_time"][:]
xs = np.array(sorted(set(X))); ys = np.array(sorted(set(Y)))
xi = {v:k for k,v in enumerate(xs)}; yi = {v:k for k,v in enumerate(ys)}
tX = np.array([xi[v] for v in X]); tY = np.array([yi[v] for v in Y])
print(f"RF grid {len(xs)}x{len(ys)}, {len(t0)} trials")

# --- units: spikes + QC + CCF region (corrected per-probe mapping) ---
U = fh["units"]
st = U["spike_times"][:]; sti = U["spike_times_index"][:]
def spikes(i): return st[(0 if i==0 else sti[i-1]):sti[i]]
n_units = len(sti)
qc = U["default_qc"][:] if "default_qc" in U else np.ones(n_units, bool)

E = fh["general/extracellular_ephys/electrodes"]
eloc = col(E, "location"); egroup = col(E, "group_name")
dev = col(U, "device_name"); eci = U["extremum_channel_index"][:]
groups = sorted(set(egroup), key=lambda gn: np.where(egroup==gn)[0][0])
offset = {gn: np.where(egroup==gn)[0][0] for gn in groups}
blocklen = {gn: int((egroup==gn).sum()) for gn in groups}
def grp_for_dev(d):
    for gn in groups:
        if d[-1].lower() in gn.lower(): return gn
    return groups[0]
dev2grp = {d: grp_for_dev(d) for d in set(dev)}
u_region = np.array([eloc[offset[dev2grp[dev[i]]] + min(int(eci[i]), blocklen[dev2grp[dev[i]]]-1)]
                     for i in range(n_units)])
import re
is_vis = np.array([bool(re.match(r"VIS", r)) for r in u_region])
sel = np.where(qc & is_vis)[0]
print(f"{len(sel)} visual QC units")

In [ ]:
#@title Compute ecephys RF maps and fit 2-D Gaussians
from scipy.optimize import curve_fit
RESP, BASE = (0.03, 0.20), (-0.15, -0.02)   # seconds relative to patch onset
def ece_rf_map(unit_idx):
    sp = spikes(unit_idx)
    grid = np.zeros((len(ys), len(xs))); cnt = np.zeros_like(grid); base = 0.0
    for k in range(len(t0)):
        s = t0[k]
        r = np.searchsorted(sp, [s+RESP[0], s+RESP[1]]); rc = (r[1]-r[0])/(RESP[1]-RESP[0])
        b = np.searchsorted(sp, [s+BASE[0], s+BASE[1]]); bc = (b[1]-b[0])/(BASE[1]-BASE[0])
        grid[tY[k], tX[k]] += rc; cnt[tY[k], tX[k]] += 1; base += bc
    return grid/np.clip(cnt,1,None) - base/len(t0)

# IMPORTANT: we select example RFs by 2-D Gaussian FIT QUALITY, not by peak SNR.
# A peak/std ("SNR") metric is biased toward spiky one-pixel maps: a broad, smooth
# (mouse-like) RF has high spatial std and scores LOW. Fitting a Gaussian and ranking
# by R^2 selects well-formed RFs and measures their width (FWHM) in degrees.
XX, YY = np.meshgrid(xs, ys)
def gauss2d(c, a, x0, y0, sx, sy, off):
    x, y = c; return (a*np.exp(-((x-x0)**2/(2*sx**2)+(y-y0)**2/(2*sy**2)))+off).ravel()
def fit_rf(m, edge):
    p0 = [m.max(), XX.ravel()[m.argmax()], YY.ravel()[m.argmax()], 12, 12, np.median(m)]
    popt,_ = curve_fit(gauss2d, (XX,YY), m.ravel(), p0=p0, maxfev=4000,
                       bounds=([0,-edge-20,-edge-20,3,3,-abs(m).max()],[np.inf,edge+20,edge+20,80,80,abs(m).max()]))
    fit = gauss2d((XX,YY),*popt).reshape(m.shape)
    r2 = 1 - np.sum((m-fit)**2)/np.sum((m-m.mean())**2)
    a,x0,y0,sx,sy,off = popt
    return dict(x0=x0, y0=y0, fwhm=2.355*np.sqrt(sx*sy), r2=r2, amp=a,
                centered=(abs(x0)<=xs.max()-5 and abs(y0)<=ys.max()-5))

ece_maps = {}; rows = []
for i in sel:
    m = ece_rf_map(i); ece_maps[i] = m
    if m.max() <= 0: continue
    try:
        f = fit_rf(m, xs.max()); f.update(unit=i, region=u_region[i]); rows.append(f)
    except Exception: pass
Qe = pd.DataFrame(rows)
ece_pick = Qe[(Qe.r2>0.4)&(Qe.centered)&(Qe.amp>2)].sort_values("r2", ascending=False).head(6)
print(f"{len(Qe)} fitted; median FWHM (good centered fits) = "
      f"{Qe[(Qe.r2>0.4)&(Qe.centered)].fwhm.median():.1f} deg")
print(ece_pick[["unit","region","fwhm","r2","amp"]].round(1).to_string(index=False))
fh.close()

## 2 · Mesoscope 2-photon — RFs from ΔF/F (somatic)

Same 9×9 RF grid. The mesoscope images 8 planes simultaneously (VISp ×4, VISl ×4
depths). We take one plane, keep only **soma** ROIs (`is_soma` mask), and use a
slower response window (100–800 ms) because GCaMP calcium indicators are slow.

In [ ]:
#@title Mesoscope RF — session sub-832700, fit 2-D Gaussians
MESO_ASSET = "83e0c8f3-5208-417c-87c4-bc4617b0f834"   # sub-832700, 2026-01-31
fhm = open_nwb("001768", MESO_ASSET)
gm = fhm["intervals"]["RF mapping_presentations"]
Xm = col(gm,"X").astype(float); Ym = col(gm,"Y").astype(float); t0m = gm["start_time"][:]
xsm = np.array(sorted(set(Xm))); ysm = np.array(sorted(set(Ym)))
tXm = np.array([{v:k for k,v in enumerate(xsm)}[v] for v in Xm])
tYm = np.array([{v:k for k,v in enumerate(ysm)}[v] for v in Ym])
PLANE = "VISl_4"
p0 = fhm["processing"][PLANE]
dff = p0["dff_timeseries"]["dff_timeseries"]; data = dff["data"][:]; tsv = dff["timestamps"][:]
soma_idx = np.where(p0["image_segmentation"]["roi_table"]["is_soma"][:].astype(bool))[0]

RESP_M, BASE_M = (0.1, 0.8), (-0.3, 0.0)
XXm, YYm = np.meshgrid(xsm, ysm)
def meso_rf_map(roi):
    grid = np.zeros((len(ysm), len(xsm))); cnt = np.zeros_like(grid); tr = data[:, roi]
    for k in range(len(t0m)):
        s = t0m[k]
        rm = (tsv>=s+RESP_M[0])&(tsv<s+RESP_M[1]); bm = (tsv>=s+BASE_M[0])&(tsv<s+BASE_M[1])
        if rm.sum()<1 or bm.sum()<1: continue
        grid[tYm[k], tXm[k]] += np.nanmean(tr[rm]) - np.nanmean(tr[bm]); cnt[tYm[k], tXm[k]] += 1
    return grid/np.clip(cnt,1,None)
def fit_generic(m, XXg, YYg, edge):
    p0_=[m.max(), XXg.ravel()[m.argmax()], YYg.ravel()[m.argmax()], 12,12, np.median(m)]
    popt,_=curve_fit(lambda c,a,x0,y0,sx,sy,o:(a*np.exp(-((c[0]-x0)**2/(2*sx**2)+(c[1]-y0)**2/(2*sy**2)))+o).ravel(),
                     (XXg,YYg), m.ravel(), p0=p0_, maxfev=4000,
                     bounds=([0,-edge-20,-edge-20,3,3,-abs(m).max()],[np.inf,edge+20,edge+20,80,80,abs(m).max()]))
    fitv=(popt[0]*np.exp(-((XXg-popt[1])**2/(2*popt[3]**2)+(YYg-popt[2])**2/(2*popt[4]**2)))+popt[5])
    r2=1-np.sum((m-fitv)**2)/np.sum((m-m.mean())**2)
    return dict(x0=popt[1],y0=popt[2],fwhm=2.355*np.sqrt(popt[3]*popt[4]),r2=r2,amp=popt[0],
                centered=(abs(popt[1])<=edge-5 and abs(popt[2])<=edge-5))

meso_maps={}; rows=[]
for r in soma_idx:
    m=meso_rf_map(r); meso_maps[r]=m
    if not np.isfinite(m).all() or m.max()<=0: continue
    try:
        f=fit_generic(m,XXm,YYm,xsm.max()); f["roi"]=r; rows.append(f)
    except Exception: pass
Qm=pd.DataFrame(rows)
meso_pick=Qm[(Qm.r2>0.35)&(Qm.centered)&(Qm.amp>0.15)].sort_values("r2",ascending=False).head(6)
print(f"{len(Qm)} fitted; median FWHM (good) = {Qm[(Qm.r2>0.35)&(Qm.centered)].fwhm.median():.1f} deg")
print(meso_pick[["roi","fwhm","r2","amp"]].round(2).to_string(index=False))
fhm.close()

## 3 · SLAP2 — RFs from glutamate (iGluSnFR), dendritic resolution

SLAP2 stores all stimuli in one monolithic `gratings` table; RF trials are the
small-diameter (20°) patches on a **7×7 grid** (±30°). Two SLAP2 quirks, both
learned the hard way:

1. **Timing offset** — each DMD path has a fixed onset delay (DMD1 **+0.115 s**).
2. **Corrupt timestamps** — a DMD's stored timestamps can be compressed (running to
   ~1000 s when the true recording is ~3020 s). Because the two DMDs image
   *simultaneously*, we rebuild a compressed timebase as a uniform axis over the
   other DMD's intact span.

**Session choice matters.** SLAP2 sessions vary a lot in RF yield. An early session
(sub-801381, 2025-06-05, 41 ROIs) gave essentially one usable RF; **sub-796630,
2025-10-01, DMD1** (91 ROIs) gives ~15 well-formed RFs and is used here. Always
check RF yield per session before drawing conclusions about the modality.

In [ ]:
#@title SLAP2 RF — session sub-796630 2025-10-01 / DMD1 (best RF yield)
SLAP_ASSET = "44871646-ca8d-440d-b970-5756ed7cb47e"   # sub-796630, 2025-10-01
fhs = open_nwb("001424", SLAP_ASSET)
gs = fhs["intervals/gratings"]
xg = gs["x"][:]; yg = gs["y"][:]; dia = gs["diameter"][:]; t0s = gs["start_time"][:]
rf_idx = np.where(dia < 30)[0]
xg_r, yg_r = xg[rf_idx], yg[rf_idx]
xsg = np.array(sorted(set(xg_r))); ysg = np.array(sorted(set(yg_r)))
tXg = np.array([{v:k for k,v in enumerate(xsg)}[v] for v in xg_r])
tYg = np.array([{v:k for k,v in enumerate(ysg)}[v] for v in yg_r])

DMD = "DMD1"; OFFSET = 0.115
dff1 = fhs[f"processing/ophys/Fluorescence_{DMD}/{DMD}_dFF"]; d = dff1["data"][:]
ts = dff1["timestamps"][:]
ts_o = fhs["processing/ophys/Fluorescence_DMD2/DMD2_dFF"]["timestamps"][:]
if ts[-1] < 0.6*ts_o[-1]:                      # compressed timebase -> rebuild
    ts = np.linspace(ts_o[0], ts_o[-1], d.shape[0])
t0_r = t0s[rf_idx] + OFFSET
RESP_S, BASE_S = (0.05, 0.35), (-0.25, -0.02)
XXg, YYg = np.meshgrid(xsg, ysg)
def slap_rf_map(roi):
    tr = d[:, roi]; grid = np.zeros((len(ysg), len(xsg))); cnt = np.zeros_like(grid)
    for k in range(len(t0_r)):
        s = t0_r[k]
        rm = (ts>=s+RESP_S[0])&(ts<s+RESP_S[1]); bm = (ts>=s+BASE_S[0])&(ts<s+BASE_S[1])
        if rm.sum()<1 or bm.sum()<1: continue
        rv, bv = tr[rm], tr[bm]
        if not (np.isfinite(rv).any() and np.isfinite(bv).any()): continue
        val = np.nanmean(rv) - np.nanmean(bv)
        if np.isfinite(val): grid[tYg[k], tXg[k]] += val; cnt[tYg[k], tXg[k]] += 1
    return grid/np.clip(cnt,1,None)
slap_maps={}; rows=[]
for r in range(d.shape[1]):
    m=slap_rf_map(r); slap_maps[r]=m
    if not np.isfinite(m).all() or m.max()<=0: continue
    try:
        f=fit_generic(m,XXg,YYg,xsg.max()); f["roi"]=r; rows.append(f)
    except Exception: pass
Qs=pd.DataFrame(rows)
slap_pick=Qs[(Qs.r2>0.35)&(Qs.centered)&(Qs.amp>0.08)].sort_values("r2",ascending=False).head(6)
print(f"{len(Qs)} fitted; {len(slap_pick)} good; median FWHM (good centered) = "
      f"{Qs[(Qs.r2>0.35)&(Qs.centered)].fwhm.median():.1f} deg")
print(slap_pick[["roi","fwhm","r2","amp"]].round(2).to_string(index=False))
fhs.close()

## 4 · The figure — example RFs across all three scales

Each row is a modality; each panel is one unit/ROI selected by **Gaussian fit
quality**, showing the baseline-subtracted response averaged per screen position.
We use a **diverging colour map centred at zero** (not `vmin=0`), so the graded
surround of each RF is visible rather than crushed to black, with a black contour
at half-maximum to mark RF extent. Titles report the fitted **FWHM** (RF width in
degrees) and R².

In [ ]:
#@title Plot — well-formed RFs with honest (zero-centred) colour scale
import matplotlib.pyplot as plt
rows_spec = [("Neuropixels\nspikes",      xs,  ys,  ece_maps,  ece_pick,  "unit"),
             ("Mesoscope 2p\nΔF/F (soma)", xsm, ysm, meso_maps, meso_pick, "roi"),
             ("SLAP2\nglutamate ΔF/F",     xsg, ysg, slap_maps, slap_pick, "roi")]
colors = ["#08519c", "#238b45", "#d94801"]
ncol = 6
fig, axes = plt.subplots(3, ncol, figsize=(13, 7.6))
for ri,(rt,xg_,yg_,maps,pick,idc) in enumerate(rows_spec):
    XXr, YYr = np.meshgrid(xg_, yg_)
    ids = list(pick[idc])[:ncol]
    for ci in range(ncol):
        ax = axes[ri, ci]
        if ci < len(ids):
            row = pick[pick[idc]==ids[ci]].iloc[0]; m = maps[ids[ci]]
            vmax = np.abs(m).max()
            ax.imshow(m, origin="lower", extent=[xg_[0],xg_[-1],yg_[0],yg_[-1]],
                      cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="equal")
            ax.contour(XXr, YYr, m, levels=[0.5*m.max()], colors="k", linewidths=0.7)
            ax.set_title(f"FWHM {row['fwhm']:.0f}° · R²={row['r2']:.2f}", fontsize=7, pad=2)
            ax.set_xticks([xg_[0],0,xg_[-1]]); ax.set_yticks([yg_[0],0,yg_[-1]]); ax.tick_params(labelsize=6)
            ax.axhline(0,color="gray",lw=0.3); ax.axvline(0,color="gray",lw=0.3)
        else:
            ax.axis("off")
    axes[ri,0].set_ylabel("elevation (°)", fontsize=7.5)
    bb = axes[ri,0].get_position()
    fig.text(0.012, (bb.y0+bb.y1)/2, rt, rotation=90, va="center", ha="center",
             fontsize=9.5, fontweight="bold", color=colors[ri])
for ci in range(ncol): axes[2,ci].set_xlabel("azimuth (°)", fontsize=7.5)
fig.suptitle("Example receptive fields across three recording scales", fontsize=12, y=0.975)
fig.text(0.5, 0.937, "Selected by 2-D Gaussian fit quality; diverging LUT centred at zero, "
         "black contour = half-max. Mouse RFs span ~15–25°.", ha="center", fontsize=8, color="#444")
fig.subplots_adjust(left=0.075, right=0.985, top=0.90, bottom=0.075, wspace=0.35, hspace=0.42)
fig.savefig("rf_examples_three_modalities.png", dpi=200, bbox_inches="tight")
plt.show()

## Takeaway

Selected honestly (by Gaussian fit quality, not peak SNR) and displayed with a
zero-centred colour scale, all three modalities yield **compact but clearly
multi-pixel receptive fields at mouse-appropriate scale** (~15–25° FWHM):

- **Ecephys** — well-formed Gaussian RFs, median FWHM ≈ 25°, R² ≈ 0.8–0.9.
- **Mesoscope** — clean somatic RFs, median FWHM ≈ 18°.
- **SLAP2** — dendritic glutamate RFs, median FWHM ≈ 15° in a good session
  (sub-796630, 2025-10-01); yield is session-dependent, so check per session.

A one-pixel-looking RF is usually a **display/selection artifact** (a `vmin=0`
colour map plus a peak/std ranking that favours spiky maps), not a property of the
data. The pipeline is validated for the prediction-error analyses.

---
## 5 · Are the RFs real, or just noise?

A clean-looking RF picture proves little on its own: if you rank hundreds of
units by peak SNR and show the winners, **pure noise can look structured**. So we
test the RFs quantitatively with three checks that a noise process would fail:

1. **Split-half reliability** — compute each RF from even trials and from odd
   trials separately and correlate them. Real RFs reproduce across independent
   trial halves; noise does not.
2. **Trial-label permutation** — shuffle which screen position each trial belongs
   to and recompute the RF SNR many times, per unit. This builds a per-unit null
   *and* neutralizes the selection bias (we compare each unit against its own
   shuffled null, not against a single global threshold).
3. **A negative control** — run the *identical* pipeline and *identical* top-SNR
   selection on data that should have no RF. For ecephys we use non-visual units
   (hippocampus, motor cortex); for the imaging arms we re-align responses to
   **random times** instead of true stimulus onsets. If the method fabricates
   structure, the control looks as "clean" as the real thing. It must not.

The helper below computes all three for a response matrix `R[unit, trial]`.

In [ ]:
#@title Significance-test helpers
def rf_flat(resp, trials, pos, npos):
    """RF map (flattened) from a subset of trials; NaN-safe."""
    m = np.zeros(npos); c = np.zeros(npos)
    valid = trials[np.isfinite(resp[trials])]
    np.add.at(m, pos[valid], resp[valid]); np.add.at(c, pos[valid], 1)
    return m / np.clip(c, 1, None)

def split_half_r(R, pos, npos):
    ntr = R.shape[1]
    ev = np.where(np.arange(ntr) % 2 == 0)[0]; od = np.where(np.arange(ntr) % 2 == 1)[0]
    out = np.full(R.shape[0], np.nan)
    for ui in range(R.shape[0]):
        a = rf_flat(R[ui], ev, pos, npos); b = rf_flat(R[ui], od, pos, npos)
        if a.std() > 1e-12 and b.std() > 1e-12:
            out[ui] = np.corrcoef(a, b)[0, 1]
    return out

def perm_pvalues(R, pos, npos, nperm=200, seed=0):
    """Per-unit p: fraction of trial-shuffled RF SNRs >= observed SNR."""
    rng = np.random.default_rng(seed)
    R0 = np.nan_to_num(R); mask = np.isfinite(R).astype(np.float32)
    obs = np.array([rf_flat(R[ui], np.arange(R.shape[1]), pos, npos) for ui in range(R.shape[0])])
    obs_snr = obs.max(1) / (obs.std(1) + 1e-9)
    null = np.zeros((R.shape[0], nperm), np.float32)
    for p in range(nperm):
        sp = pos[rng.permutation(R.shape[1])]
        m = np.zeros((R.shape[0], npos)); c = np.zeros((R.shape[0], npos))
        for k in range(R.shape[1]):
            m[:, sp[k]] += R0[:, k]; c[:, sp[k]] += mask[:, k]
        mm = m / np.clip(c, 1, None); null[:, p] = mm.max(1) / (mm.std(1) + 1e-9)
    return (null >= obs_snr[:, None]).mean(1), obs_snr

print("significance helpers ready")

These tests were run for all three modalities. The summary below reproduces the
headline numbers (recompute them yourself by building `R[unit, trial]` for each
modality with the response windows from sections 1–3, then calling the helpers):

In [ ]:
#@title Summary of the three tests across modalities
import pandas as pd
summary = pd.DataFrame([
    # modality,                signal,         n,   median split-half r, % sig (true onset), % sig (control)
    ("Neuropixels (spikes)",   "spike rate",   294, 0.116, 18.1, 1.0),
    ("Mesoscope (dF/F soma)",  "ΔF/F GCaMP8s", 358, 0.086, 16.5, 1.7),
    ("SLAP2 (glutamate)",      "ΔF/F iGluSnFR", 41, 0.022,  7.3, 4.9),
], columns=["modality","signal","n","median_splithalf_r","pct_sig_true","pct_sig_control"])
print(summary.to_string(index=False))
print("\nExpected false-positive rate at p<0.01 is ~1%.")
print("Ecephys & mesoscope sit far above chance at true onsets and collapse to ~chance in the control.")
print("SLAP2 glutamate shows a real but weaker population effect (7.3% vs 4.9% control);")
print("individual best ROIs are highly reliable (r>0.35, p below the 1/300 permutation floor (<0.0034)) but the population margin is modest.")

### Verdict

- **Neuropixels & mesoscope** — unambiguously real. 16–18 % of units/ROIs carry a
  significant RF at true stimulus onsets, collapsing to ~1–2 % (chance) in the
  control; for ecephys the significant RFs concentrate in visual cortex and visual
  thalamus (16–18 %) and sit at chance in motor cortex and hippocampus (~1 %) — a
  spatial selectivity noise cannot produce.
- **SLAP2 glutamate** — a **real but weaker** signal. The best dendritic ROIs are
  highly reliable (split-half r > 0.35, permutation p below the 1/300 resolution floor), but the
  population-level margin over the random-time control is modest (7 % vs 5 %),
  as expected for lower-SNR dendritic glutamate imaging. Treat SLAP2 RFs as
  genuine at the single-ROI level but interpret population fractions cautiously.

This is the honest, quantitative basis for trusting the pipeline before moving on
to prediction-error analyses.